<a href="https://colab.research.google.com/github/centbenjamin6-dot/Health-informatics-course/blob/main/N2_2_loops_and_functions_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# N2.2 · Loops and Functions

**Day 2 · about 2 hours** · *Health Analytics and Sustainability (ESG) course*

**You are using Google Colab.** Notebooks open in your browser and run on Google's computers, so there is nothing to install. If Colab says the session has disconnected, click **Reconnect** and run the cells again from the top (Runtime → Run all).

### 🎯 What you will be able to do after this notebook
- Repeat actions with **for** and **while** loops
- Build totals, counts and new lists inside a loop
- Write **functions** with parameters, a docstring and a `return` value
- Test functions with **assert** and with the course checker
- Calculate **eGFR (CKD-EPI 2021)** and replace Day 1's copy-and-paste with a loop and functions

### 🗺️ Roadmap
| Part | Topic | Time |
|---|---|---|
| A | `for` loops over lists, `range`, `zip` | 30 min |
| B | Totals, counts and building lists | 20 min |
| C | `while` loops | 10 min |
| D | Functions: `def` and `return` | 30 min |
| E | Clinical functions: BMI, glucose, creatinine, eGFR, CKD stage | 20 min |
| F | Mini-project: the clinic report | 10 min |

⚠️ All clinical formulas and thresholds here are **educational**, not for clinical use.

**Reminder:** run cells with **Shift + Enter**, in order. `____` means *fill in the blank*. Legend: 📘 read · ▶️ run · ✏️ fill in · 🧑‍💻 your turn · 🎯 check · 💡 hint.

## Setup
▶️ Run the checker cell once. This version can also **test functions** (you will see it in Part D).

In [101]:
# 🧰 COURSE HELPER: just run this cell once. (You will understand every line by Day 2.)
_results = {}

def _report(name, ok, message):
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + message)

def check(name, expected, tol=0, hint=""):
    """Is the variable called `name` equal to the expected answer?"""
    if name not in globals():
        _report(name, False, f"{name}: I can't find this variable yet. Fill in the blank or write your code, then run the cell again. {hint}".strip())
        return
    value = globals()[name]
    try:
        if isinstance(expected, bool):
            ok = isinstance(value, bool) and value == expected
        elif isinstance(expected, (int, float)):
            ok = (not isinstance(value, bool)) and abs(value - expected) <= tol
        else:
            ok = value == expected
    except Exception:
        ok = False
    if ok:
        _report(name, True, f"{name}: correct!")
    else:
        _report(name, False, f"{name}: not quite yet. Your value is {value!r}. {hint}".strip())

def check_type(name, expected_type, hint=""):
    """Does the variable called `name` hold the expected type of value?"""
    if name not in globals():
        _report(name, False, f"{name}: I can't find this variable yet. {hint}".strip())
        return
    value = globals()[name]
    ok = type(value) is expected_type
    if ok:
        _report(name, True, f"{name}: correct type ({expected_type.__name__})!")
    else:
        _report(name, False, f"{name}: expected type {expected_type.__name__}, but yours is {type(value).__name__}. {hint}".strip())

def check_that(name, test, hint=""):
    """Does the value in `name` pass a custom test?"""
    if name not in globals():
        _report(name, False, f"{name}: I can't find this variable yet. {hint}".strip())
        return
    value = globals()[name]
    try:
        ok = bool(test(value))
    except Exception:
        ok = False
    if ok:
        _report(name, True, f"{name}: correct!")
    else:
        _report(name, False, f"{name}: not quite yet. Your value is {value!r}. {hint}".strip())

def progress():
    passed, total = sum(_results.values()), len(_results)
    print(f"You have passed {passed} of {total} checks so far.")
    if total > 0 and passed == total:
        print("🎉 All checks passed. Well done!")


def check_function(name, cases, tol=0, hint=""):
    """Call the function called `name` with test inputs and compare the results with the expected outputs."""
    if name not in globals() or not callable(globals()[name]):
        _report(name, False, f"{name}: I can't find a function with this name yet. {hint}".strip())
        return
    func = globals()[name]
    for args, expected in cases:
        try:
            result = func(*args)
            if isinstance(expected, bool):
                ok = isinstance(result, bool) and result == expected
            elif isinstance(expected, (int, float)):
                ok = (not isinstance(result, bool)) and result is not None and abs(result - expected) <= tol
            else:
                ok = result == expected
        except Exception:
            ok, result = False, "(an error)"
        if not ok:
            _report(name, False, f"{name}{args}: your function gave {result!r}, which is not what I expected. {hint}".strip())
            return
    _report(name, True, f"{name}: passed all {len(cases)} test cases!")

## Part A · `for` loops: do something for every item
📘 A **loop** repeats a block of code. A **`for` loop** goes through a list **one item at a time**.
```
for weight in weights_kg:
│   │      │  │
│   │      │  └── the list to go through
│   │      └── the word "in"
│   └── a name for "the current item" (you choose it; it changes on every round)
└── keyword for ... and a COLON at the end, as with if
    print(weight)      ← indented: this block runs ONCE PER ITEM
```
### ▶️ Worked example A1: print every weight

In [102]:
weights_kg = [62.5, 80.1, 71.3, 90.4, 55.0]

for weight in weights_kg:
    print("Weight:", weight, "kg")

print("Loop finished")           # not indented: runs once, after the loop

Weight: 62.5 kg
Weight: 80.1 kg
Weight: 71.3 kg
Weight: 90.4 kg
Weight: 55.0 kg
Loop finished


### 📘 Three handy partners of the `for` loop
| Tool | What it gives | Example |
|---|---|---|
| `range(5)` | the numbers 0, 1, 2, 3, 4 | `for i in range(5):` |
| `range(1, 6)` | 1, 2, 3, 4, 5 (stops **before** 6) | `for day in range(1, 6):` |
| `zip(a, b)` | pairs from two lists, matched by position | `for w, h in zip(weights, heights):` |

### ▶️ Worked example A2: `range` and `zip`

In [103]:
print(list(range(5)))               # list() lets us see the numbers: [0, 1, 2, 3, 4]

for day in range(1, 4):
    print("Day", day)

weights_kg = [62.5, 80.1, 71.3, 90.4, 55.0]
heights_m  = [1.60, 1.80, 1.75, 1.85, 1.58]

for w, h in zip(weights_kg, heights_m):       # w and h always belong to the SAME patient
    print(f"{w} kg, {h} m -> BMI {w / h ** 2:.1f}")

[0, 1, 2, 3, 4]
Day 1
Day 2
Day 3
62.5 kg, 1.6 m -> BMI 24.4
80.1 kg, 1.8 m -> BMI 24.7
71.3 kg, 1.75 m -> BMI 23.3
90.4 kg, 1.85 m -> BMI 26.4
55.0 kg, 1.58 m -> BMI 22.0


## Part B · Totals, counts and building lists
📘 Three patterns cover most loops:

| Pattern | Idea | Skeleton |
|---|---|---|
| **Accumulate** | start at 0, add each item | `total = 0` → `total = total + x` |
| **Count** | start at 0, add 1 when a condition is True | `if x > 80: count = count + 1` |
| **Collect** | start with `[]`, `.append()` each result | `results = []` → `results.append(...)` |

⚠️ The starting line (`total = 0`) goes **before** the loop, not inside it.

### ▶️ Worked example B1: accumulate, count, collect

In [104]:
doses_mg = [250, 500, 250, 750]

total = 0                        # accumulate
count_big = 0                    # count
doubled = []                     # collect
for dose in doses_mg:
    total = total + dose
    if dose >= 500:
        count_big = count_big + 1
    doubled.append(dose * 2)

print(total, count_big, doubled)      # 1750 2 [500, 1000, 500, 1500]

1750 2 [500, 1000, 500, 1500]


### ✏️ Exercise 2.2.1 · Total and mean weight
Fill in the blank so the loop adds **each weight** to `total_weight`.

In [105]:
weights_kg = [62.5, 80.1, 71.3, 90.4, 55.0]

total_weight = 0
for w in weights_kg:
    total_weight = total_weight + w

mean_weight = total_weight / len(weights_kg)

🎯 **Check**

In [106]:
check("total_weight", 359.3, tol=0.001)
check("mean_weight", 71.86, tol=0.001)

✅ total_weight: correct!
✅ mean_weight: correct!


### ✏️ Exercise 2.2.2 · Count the fast heart rates
Count how many readings are **above 80**. Fill in the threshold.

In [107]:
heart_rates = [72, 88, 65, 90, 78]

count_high = 0
for rate in heart_rates:
    if rate > 80:
        count_high = count_high + 1
        print(count_high)

1
2


🎯 **Check**

In [108]:
check("count_high", 2)

✅ count_high: correct!


### 🧑‍💻 Exercise 2.2.3 · A list of BMIs
Build a list `bmi_list` that contains the **BMI of each patient rounded to 1 decimal**, using `zip` over the two lists. Use the collect pattern: start with an empty list, then `.append()` inside the loop.

In [109]:
weights_kg = [62.5, 80.1, 71.3, 90.4, 55.0]
heights_m  = [1.60, 1.80, 1.75, 1.85, 1.58]

# Write your code here
weights_kg = [62.5, 80.1, 71.3, 90.4, 55.0]
heights_m = [1.60, 1.80, 1.75, 1.85, 1.58]

bmi_list = []

for w, h in zip(weights_kg, heights_m):
    bmi = round(w / (h ** 2), 1)
    bmi_list.append(bmi)

print(bmi_list)



[24.4, 24.7, 23.3, 26.4, 22.0]


💡 <details><summary>Hint</summary>Pattern: <code>bmi_list = []</code> first, then <code>for w, h in zip(weights_kg, heights_m):</code> and inside the loop <code>bmi_list.append(round(w / h ** 2, 1))</code>.</details>

🎯 **Check**

In [110]:
check("bmi_list", [24.4, 24.7, 23.3, 26.4, 22.0], hint="Round each BMI to 1 decimal and keep the order of the patients.")

✅ bmi_list: correct!


### ▶️ Worked example B2: looping over a list of dictionaries
A `for` loop works on records too: each round, `p` is one patient's dictionary.

In [111]:
patients = [
    {"id": "P001", "age": 54, "sex": "F", "weight_kg": 68.0, "height_cm": 165, "glucose_mgdl": 126, "creatinine_umol_l": 88},
    {"id": "P002", "age": 61, "sex": "M", "weight_kg": 82.5, "height_cm": 178, "glucose_mgdl": 98,  "creatinine_umol_l": 110},
    {"id": "P003", "age": 47, "sex": "F", "weight_kg": 59.0, "height_cm": 160, "glucose_mgdl": 182, "creatinine_umol_l": 70},
]

for p in patients:
    print(p["id"], "is", p["age"], "years old")

P001 is 54 years old
P002 is 61 years old
P003 is 47 years old


### ✏️ Exercise 2.2.4 · Mean age
Add up the ages of the patients. Fill in the **key** that gives the age.

In [112]:
total_age = 0
for p in patients:
    total_age = total_age + p["age"]

mean_age = total_age / len(patients)

🎯 **Check**

In [113]:
check("total_age", 162)
check("mean_age", 54.0, tol=0.001)

✅ total_age: correct!
✅ mean_age: correct!


## Part C · `while` loops: repeat until something changes
📘 A **`while` loop** keeps repeating **as long as its condition is True**. Use it when you do not know in advance how many rounds are needed.
```
while concentration > 10:
│     │
│     └── checked BEFORE every round; the loop stops when it becomes False
└── keyword while ... COLON at the end
    concentration = concentration / 2     ← something inside MUST change the condition
```
⚠️ **If nothing in the loop can make the condition False, it runs forever** (an *infinite loop*). In that case press the **Stop ■** button (Colab) or **Kernel → Interrupt** (Jupyter).

### ▶️ Worked example C1: drug half-lives
A drug level of 100 mg halves every 6 hours. How long until it is below 10 mg?

In [114]:
level_mg = 100.0
elapsed_h = 0

while level_mg > 10:
    level_mg = level_mg / 2       # one half-life passes
    elapsed_h = elapsed_h + 6

print(f"After {elapsed_h} hours the level is {level_mg} mg")       # 24 hours, 6.25 mg

After 24 hours the level is 6.25 mg


### ✏️ Exercise 2.2.5 · A faster drug
A drug level starts at **200 mg** and **halves every 4 hours**. Loop **while the level is above 10 mg**. Fill in the two numbers.

In [115]:
concentration = 200.0
hours = 0

while concentration > 10:
    concentration = concentration / 2
    hours = hours + 4

🎯 **Check**

In [116]:
check("hours", 20)
check("concentration", 6.25, tol=0.001)

✅ hours: correct!
✅ concentration: correct!


## Part D · Functions: name a recipe once, reuse it forever
📘 A **function** is a named, reusable piece of code. You have used built-in ones (`print`, `len`, `round`). Now you write your own with **`def`** (short for *define*).
```
def calculate_bsa(height_cm, weight_kg):
│   │             │                    │
│   │             └────────────────────┘ PARAMETERS: the inputs (names you choose)
│   └── the function name (lowercase_with_underscores; a verb is a good start)
└── keyword def ... and a COLON at the end
    """Return body surface area in m² (Mosteller)."""   ← DOCSTRING: one line saying what it does
    bsa = (height_cm * weight_kg / 3600) ** 0.5     ← the body is INDENTED
    return bsa                                        ← return: hand the answer back
```
### Defining versus calling
- **Defining** a function (`def ...`) only stores the recipe. Nothing happens yet.
- **Calling** it (`calculate_bsa(180, 80)`) runs the recipe with those inputs and gives back the `return` value.

### `print` versus `return`: the classic confusion
- `print(x)` **shows** something on the screen, but the function hands nothing back.
- `return x` **hands the value back** so you can store it, calculate with it, or test it.

### ▶️ Worked example D1: your first function

In [117]:
def calculate_bsa(height_cm, weight_kg):
    """Return body surface area in m² (Mosteller formula)."""
    bsa = (height_cm * weight_kg / 3600) ** 0.5
    return bsa

# Calling the function with different inputs:
print(calculate_bsa(180, 80))                       # 2.0
print(round(calculate_bsa(165, 60), 2))             # 1.66
patient_bsa = calculate_bsa(170, 70)                # store the returned value
print(f"BSA {patient_bsa:.2f} m²")

2.0
1.66
BSA 1.82 m²


### ▶️ Worked example D2: print versus return
Run this and compare the two results.

In [118]:
def double_with_print(x):
    print(x * 2)            # shows the result, returns nothing

def double_with_return(x):
    return x * 2            # hands the result back

a = double_with_print(5)     # prints 10
b = double_with_return(5)    # prints nothing

print("a is", a)            # None: print gave nothing back
print("b is", b)            # 10: we can use it

10
a is None
b is 10


### ▶️ Worked example D3: a function that decides
A function can contain `if / elif / else` with several `return` lines. The **first `return` reached ends the function**.

In [119]:
def glucose_category_of(glucose_mgdl):
    """Return the fasting-glucose category as text."""
    if glucose_mgdl < 100:
        return "Normal"
    elif glucose_mgdl < 126:
        return "Prediabetes"
    else:
        return "Diabetes range"

print(glucose_category_of(90))
print(glucose_category_of(110))
print(glucose_category_of(140))

Normal
Prediabetes
Diabetes range


### 📘 How the checker tests your functions
For function exercises, the 🎯 cell **calls your function** with several test inputs and compares the answers. If one test fails it tells you **which input** gave a wrong result. Two things to remember: **write `return`**, and **keep the function name exactly as given**.

## Part E · Clinical functions
### ✏️ Exercise 2.2.6 · BMI function
Fill in the formula. Remember: **return** the value (do not print it).

In [120]:
def calculate_bmi(weight_kg, height_m):
    """Return BMI = weight (kg) divided by height (m) squared."""
    return weight_kg / (height_m ** 2)

🎯 **Check**

In [121]:
check_function("calculate_bmi", [((70, 1.75), 22.857), ((85, 1.80), 26.235), ((58, 1.65), 21.304)], tol=0.01)

✅ calculate_bmi: passed all 3 test cases!


### 🧑‍💻 Exercise 2.2.7 · BMI category function
Complete the function so that it **returns** `"Underweight"`, `"Normal weight"`, `"Overweight"` or `"Obesity"` (same thresholds as N2.1: below 18.5, below 25, below 30, otherwise). Write the body in place of the *Write your code here* line.

In [122]:
def bmi_category(bmi):
    """Return the WHO adult BMI category as text."""
def bmi_category(bmi):
    """Return the WHO adult BMI category as text."""
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal weight"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obesity"

💡 <details><summary>Hint</summary>Use <code>return</code> instead of assigning to a variable, as in Worked example D3. The checker tests boundary values such as exactly 18.5, 25 and 30.</details>

🎯 **Check**

In [123]:
check_function("bmi_category", [((17,), "Underweight"), ((18.5,), "Normal weight"), ((22.9,), "Normal weight"), ((24.99,), "Normal weight"), ((25,), "Overweight"), ((27.4,), "Overweight"), ((30,), "Obesity"), ((35,), "Obesity")], hint="Boundary values matter: 25 is Overweight, 30 is Obesity.")

✅ bmi_category: passed all 8 test cases!


### ✏️ Exercise 2.2.8 · Glucose conversion
Convert **mg/dL to mmol/L** by dividing by the factor you know from Day 1.

In [124]:
def glucose_mgdl_to_mmol(mgdl):
    """Convert glucose from mg/dL to mmol/L."""
    return mgdl / 18
    print(glucose_mgdl_to_mmol(126))

🎯 **Check**

In [125]:
check_function("glucose_mgdl_to_mmol", [((126,), 6.9938), ((90,), 4.9956)], tol=0.01)

✅ glucose_mgdl_to_mmol: passed all 2 test cases!


### 🧑‍💻 Exercise 2.2.9 · Creatinine conversion
Write a function `creatinine_umol_to_mgdl` that converts creatinine from **µmol/L to mg/dL** (divide by **88.4**).

In [126]:
def creatinine_umol_to_mgdl(umol):
    """Convert creatinine from µmol/L to mg/dL."""
    return umol / 88.4
    print(creatinine_umol_to_mgdl(88.4))



🎯 **Check**

In [127]:
check_function("creatinine_umol_to_mgdl", [((88,), 0.9955), ((176,), 1.991)], tol=0.001)

✅ creatinine_umol_to_mgdl: passed all 2 test cases!


### 📘 The eGFR equation (CKD-EPI 2021, race-free)
**eGFR** estimates how well the kidneys filter blood, from **creatinine, age and sex**. The result is in **mL/min/1.73 m²**.

**eGFR = 142 × min(Scr/κ, 1)<sup>α</sup> × max(Scr/κ, 1)<sup>−1.200</sup> × 0.9938<sup>age</sup> × (1.012 if female)**

| | Female | Male |
|---|---|---|
| κ (kappa) | 0.7 | 0.9 |
| α (alpha) | −0.241 | −0.302 |
| sex factor | 1.012 | 1.0 |

*Scr* is serum creatinine **in mg/dL** (that is why we converted units first). `min(a, b)` gives the smaller and `max(a, b)` the larger of two numbers.

### ✏️ Exercise 2.2.10 · The eGFR function
The male values are filled in. **Fill in the three female values** from the table.

In [128]:
def egfr_ckd_epi_2021(creatinine_mgdl, age, sex):
    """Estimate GFR (mL/min/1.73 m²) with the CKD-EPI 2021 equation. sex is "F" or "M"."""
    if sex == "F":
        kappa = 0.7
        alpha = -0.241
        sex_factor = 1.012
        kappa = 0.9
        alpha = -0.302
        sex_factor = 1.0
    ratio = creatinine_mgdl / kappa
    return 142 * min(ratio, 1) ** alpha * max(ratio, 1) ** -1.200 * 0.9938 ** age * sex_factor

### 📘 The eGFR equation (CKD-EPI 2021, race-free)
**eGFR** estimates how well the kidneys filter blood, from **creatinine, age and sex**. The result is in **mL/min/1.73 m²**.

**eGFR = 142 × min(Scr/κ, 1)<sup>α</sup> × max(Scr/κ, 1)<sup>−1.200</sup> × 0.9938<sup>age</sup> × (1.012 if female)**

| | Female | Male |
|---|---|---|
| κ (kappa) | 0.7 | 0.9 |
| α (alpha) | −0.241 | −0.302 |
| sex factor | 1.012 | 1.0 |

*Scr* is serum creatinine **in mg/dL** (that is why we converted units first). `min(a, b)` gives the smaller and `max(a, b)` the larger of two numbers.

### ✏️ Exercise 2.2.10 · The eGFR function
The male values are filled in. **Fill in the three female values** from the table.

🎯 **Check**

In [129]:
check_function("egfr_ckd_epi_2021", [((1.0, 54, 'F'), 66.9472), ((0.9, 40, 'M'), 110.7256), ((0.6, 30, 'F'), 123.7579), ((1.8, 70, 'M'), 39.9928)], tol=0.01)

❌ egfr_ckd_epi_2021(1.0, 54, 'F'): your function gave 89.43859038783556, which is not what I expected.


### 🧑‍💻 Exercise 2.2.11 · CKD stage from eGFR
Complete `classify_ckd_stage` using the **KDIGO GFR categories** (educational):

| eGFR | Return |
|---|---|
| 90 or more | `"G1"` |
| 60 to 89 | `"G2"` |
| 45 to 59 | `"G3a"` |
| 30 to 44 | `"G3b"` |
| 15 to 29 | `"G4"` |
| below 15 | `"G5"` |

In [140]:
def classify_ckd_stage(egfr):
    """Return the GFR category (G1 to G5) for an eGFR value."""
    if egfr >= 90:
      return "G1"
    elif egfr >= 60:
      return "G2"
    elif egfr >= 45:
      return "G3a"
    elif egfr >= 30:
      return "G3b"
    elif egfr >= 15:
      return "G4"
    else:
      return "G5"


💡 <details><summary>Hint</summary>Test from the highest threshold downwards with <code>&gt;=</code>, as in the BMI chain. Each branch uses <code>return</code>.</details>

🎯 **Check**

In [141]:
check_function("classify_ckd_stage", [((95,), "G1"), ((90,), "G1"), ((75,), "G2"), ((60,), "G2"), ((59.9,), "G3a"), ((45,), "G3a"), ((44.9,), "G3b"), ((30,), "G3b"), ((29.9,), "G4"), ((15,), "G4"), ((14.9,), "G5")], hint="Check the boundaries: 90, 60, 45, 30 and 15.")

✅ classify_ckd_stage: passed all 11 test cases!


### 📘 Testing with `assert`
`assert` is a built-in **safety net**: `assert condition` does nothing when the condition is True and raises an **`AssertionError`** when it is False. Programmers write asserts to prove that their functions behave.

### ▶️ Worked example E1: assert in action

In [142]:
assert calculate_bmi(70, 1.75) > 22            # True: nothing happens, no output
print("First assert passed")

try:
    assert calculate_bmi(70, 1.75) > 30, "BMI should be above 30"     # False: raises AssertionError
except AssertionError as error:
    print("AssertionError:", error)

First assert passed
AssertionError: BMI should be above 30


### 🧑‍💻 Exercise 2.2.12 · Write your own assert
Write **one `assert` line** that tests that `bmi_category(30)` equals `"Obesity"`. Then, on the next line, write `assert_exercise_done = True`. That second line only runs if your assert passed.

In [143]:
# Line 1: your assert
# Line 2: assert_exercise_done = True
assert bmi_category(30) == "Obesity"
assert_exercise_done = True

💡 <details><summary>Hint</summary>Use <code>==</code> (two equals signs) to compare. The assert line looks like <code>assert some_call == expected_value</code>.</details>

🎯 **Check**

In [144]:
check("assert_exercise_done", True)

✅ assert_exercise_done: correct!


## Part F · Mini-project: the clinic report
Remember Day 1's script that repeated six lines for each patient? Now a **loop** and **functions** do the same job (and more) in a few lines.

🔁 To make sure everybody continues with working tools, the next cell defines the **official** versions of all the functions from this notebook (yours should behave identically).

In [145]:
def calculate_bmi(weight_kg, height_m):
    return weight_kg / height_m ** 2

def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal weight"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obesity"

def creatinine_umol_to_mgdl(umol):
    return umol / 88.4

def egfr_ckd_epi_2021(creatinine_mgdl, age, sex):
    if sex == "F":
        kappa, alpha, sex_factor = 0.7, -0.241, 1.012
    else:
        kappa, alpha, sex_factor = 0.9, -0.302, 1.0
    ratio = creatinine_mgdl / kappa
    return 142 * min(ratio, 1) ** alpha * max(ratio, 1) ** -1.200 * 0.9938 ** age * sex_factor

def classify_ckd_stage(egfr):
    if egfr >= 90:
        return "G1"
    elif egfr >= 60:
        return "G2"
    elif egfr >= 45:
        return "G3a"
    elif egfr >= 30:
        return "G3b"
    elif egfr >= 15:
        return "G4"
    else:
        return "G5"

print("Official functions loaded ✅")

Official functions loaded ✅


### ▶️ Worked example F1: the report for one patient
Study how the functions work together: **creatinine → eGFR → stage**, **weight and height → BMI → category**.

In [136]:
p = patients[0]

bmi = calculate_bmi(p["weight_kg"], p["height_cm"] / 100)
creatinine_mgdl = creatinine_umol_to_mgdl(p["creatinine_umol_l"])
egfr = egfr_ckd_epi_2021(creatinine_mgdl, p["age"], p["sex"])

line = f"{p['id']}: BMI {bmi:.2f} ({bmi_category(bmi)}) | eGFR {egfr:.0f} ({classify_ckd_stage(egfr)})"
print(line)

P001: BMI 24.98 (Normal weight) | eGFR 67 (G2)


### 🧑‍💻 Exercise 2.2.13 · The report for every patient
Turn the worked example into a **loop** over `patients`. Build a list `clinic_lines` with **one text line per patient**, in exactly the format above, then print each line.

Steps inside the loop: (1) BMI, (2) creatinine in mg/dL, (3) eGFR, (4) build the line, (5) `append` it to `clinic_lines`.

In [149]:
clinic_lines = []

# Write your loop here
clinic_lines = []

for p in patients:
    # 1. Calculate BMI
    bmi = calculate_bmi(p["weight_kg"], p["height_cm"] / 100)

    # 2. Convert creatinine to mg/dL
    creatinine_mgdl = creatinine_umol_to_mgdl(p["creatinine_umol_l"])

    # 3. Calculate eGFR
    egfr = egfr_ckd_epi_2021(creatinine_mgdl, p["age"], p["sex"])

    # 4. Build the report line
    line = f"{p['id']}: BMI {bmi:.2f} ({bmi_category(bmi)}) | eGFR {egfr:.0f} ({classify_ckd_stage(egfr)})"

    # 5. Append the line to the list
    clinic_lines.append(line)

# Print each patient's report
for line in clinic_lines:
    print(line)

P001: BMI 24.98 (Normal weight) | eGFR 67 (G2)
P002: BMI 26.04 (Overweight) | eGFR 66 (G2)
P003: BMI 23.05 (Normal weight) | eGFR 93 (G1)


💡 <details><summary>Hint</summary>Copy the five lines of Worked example F1 into a <code>for p in patients:</code> loop (delete the line <code>p = patients[0]</code>), and add <code>clinic_lines.append(line)</code> at the end of the loop body.</details>

🎯 **Check**

In [150]:
check("clinic_lines", ['P001: BMI 24.98 (Normal weight) | eGFR 67 (G2)', 'P002: BMI 26.04 (Overweight) | eGFR 66 (G2)', 'P003: BMI 23.05 (Normal weight) | eGFR 93 (G1)'], hint="Same format as the worked example, one line per patient, in order.")

✅ clinic_lines: correct!


### 💭 Look how far you have come
In N1.3 this took about 18 lines for three patients. Now **the loop handles any number of patients** (3 or 3,000), and every calculation lives in a **tested function**. This is how real data-analysis code is organised.

## 🧾 Recap: your cheat sheet
| Idea | Syntax | Example |
|---|---|---|
| Loop over a list | `for x in items:` | `for w in weights_kg:` |
| Numbers in a row | `range(n)`, `range(a, b)` | `range(1, 6)` → 1 to 5 |
| Walk two lists together | `zip(a, b)` | `for w, h in zip(ws, hs):` |
| Total / count / collect | `t = t + x` · `c = c + 1` · `.append(x)` | |
| Repeat until | `while condition:` | `while level > 10:` |
| Define a function | `def name(params):` | `def calculate_bmi(w, h):` |
| Hand back a value | `return value` | `return w / h ** 2` |
| Describe it | `"""docstring"""` | |
| Safety net | `assert condition` | `assert bmi_category(30) == "Obesity"` |

### Common slips
- **Missing colon or indentation** after `for`, `while` and `def`.
- **Forgetting `return`** → the function gives back `None`.
- **`print` instead of `return`** inside a function.
- **Initialising inside the loop** (`total = 0` inside the loop resets it every round).
- **Infinite `while` loops:** nothing changes the condition.
- Defining a function but **never calling it**.

**Next:** N2.3, *NumPy and error handling*.

### 📊 Your progress

In [151]:
progress()   # shows how many checks you have passed

You have passed 15 of 16 checks so far.


💾 **Before you finish:** save this notebook to your GitHub repository (**File → Save a copy in GitHub**) and message your instructor that it is uploaded.